# Module 6.5 — Serving Concurrency, Batching, and Caching

Companion notebook to [`docs/modules/06_5_serving_concurrency_batching_caching.md`](../docs/modules/06_5_serving_concurrency_batching_caching.md).

Like Module 6, most of this module's infrastructure needs no live model runtime:
`FakeRuntime`'s simulated latency (Module 6) is exactly what's needed to prove queueing and
caching behavior for real. Only the real per-runtime concurrency measurement (Labs 1-3, 8
against an actual Ollama server) is honest-skip here.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_06_5"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/bhakti/workspace/local_ai_app


## 1. BoundedRequestQueue: admission, concurrency limiting, rejection

Real concurrent execution, real timing — proving max_concurrent is actually enforced.

In [2]:
import asyncio

from local_ai_core.gateway.queue import BoundedRequestQueue, QueueFullError

q = BoundedRequestQueue(max_concurrent=2, max_queue_size=5)
concurrently_running = 0
max_observed = 0

async def simulated_generation():
    global concurrently_running, max_observed
    concurrently_running += 1
    max_observed = max(max_observed, concurrently_running)
    await asyncio.sleep(0.05)
    concurrently_running -= 1
    return "generated text"

results = await asyncio.gather(*[q.submit(simulated_generation) for _ in range(6)])
print(f"max_concurrent=2 configured; observed peak concurrency = {max_observed} (must be <= 2)")
print(f"Queue waits: {[f'{r.queue_wait_seconds:.3f}s' for r in results]}")

# Admission rejection: a full queue rejects immediately, not eventually.
tight_q = BoundedRequestQueue(max_concurrent=1, max_queue_size=0)
release = asyncio.Event()

async def blocking():
    await release.wait()
    return "ok"

blocked_task = asyncio.create_task(tight_q.submit(blocking))
await asyncio.sleep(0.01)
try:
    await tight_q.submit(lambda: asyncio.sleep(0))
except QueueFullError as exc:
    print(f"\nCorrectly rejected: {exc}")
release.set()
await blocked_task

max_concurrent=2 configured; observed peak concurrency = 2 (must be <= 2)
Queue waits: ['0.000s', '0.000s', '0.051s', '0.051s', '0.103s', '0.103s']

Correctly rejected: Queue is full (0/0 already waiting)


QueuedResult(result='ok', queue_wait_seconds=1.095901825465262e-05, execution_seconds=0.010838207992492244)

## 2. Response cache and semantic cache — real hit/miss behavior

In [3]:
from local_ai_core.gateway.cache import ResponseCache, SemanticCache, response_cache_key
import numpy as np

cache = ResponseCache()
key = response_cache_key("qwen2.5:7b", "What is a KV cache?", {"temperature": 0.0}, "v1")
print(f"First lookup (should miss): {cache.get(key)}")
cache.put(key, "A KV cache stores keys/values per token per layer.")
print(f"Second lookup (should hit): {cache.get(key)}")
print(f"Hit rate: {cache.hit_rate:.0%}")

print("\n--- Semantic cache ---")
semantic = SemanticCache(similarity_threshold=0.9)
semantic.put(np.array([1.0, 0.0, 0.0]), "Cached answer about topic A", original_query="Tell me about A")
near_duplicate = np.array([0.98, 0.1, 0.0])
hit = semantic.get(near_duplicate)
print(f"Near-duplicate query hit: {hit}")
unrelated = np.array([0.0, 0.0, 1.0])
print(f"Unrelated query hit (should be None): {semantic.get(unrelated)}")

First lookup (should miss): None
Second lookup (should hit): A KV cache stores keys/values per token per layer.
Hit rate: 50%

--- Semantic cache ---
Near-duplicate query hit: SemanticCacheHit(response='Cached answer about topic A', similarity=0.9948341425312659, matched_query='Tell me about A')
Unrelated query hit (should be None): None


## 3. Lab 8 — caching before/after, with real (simulated) latency

No live runtime needed: `FakeRuntime`'s simulated latency makes this a genuine before/after
measurement, not a fabricated one.

In [4]:
import lab_caching_before_after as caching_lab

results = await caching_lab.run_lab(unique_query_count=5, repeats=4, simulated_latency_ms=20.0)
print(caching_lab.results_to_markdown(results))

# Lab 8 — caching before/after

- Workload: 20 requests, 5 unique queries
- Without cache: 0.440s (20 runtime calls)
- With cache: 0.109s (5 runtime calls)
- Speedup: 4.05x
- Cache hit rate: 75%



## 4. AdmissionPolicy: a decision on record, not a magic number

In [5]:
from local_ai_core.gateway.admission_control import AdmissionPolicy, ConcurrencyMeasurement, recommend_policy_from_measurements

default_policy = AdmissionPolicy()
print(f"Default: max_concurrent_requests={default_policy.max_concurrent_requests}")
print(f"Reason: {default_policy.reason}")

try:
    AdmissionPolicy(max_concurrent_requests=4)  # no reason given
except ValueError as exc:
    print(f"\nCorrectly rejected an unjustified higher concurrency: {exc}")

print("\n--- Deriving a policy from measurements ---")
good_measurements = [
    ConcurrencyMeasurement(concurrency=1, mean_latency_seconds=1.0, p95_latency_seconds=1.2, failure_rate=0.0),
    ConcurrencyMeasurement(concurrency=2, mean_latency_seconds=1.1, p95_latency_seconds=1.4, failure_rate=0.0),
    ConcurrencyMeasurement(concurrency=4, mean_latency_seconds=1.3, p95_latency_seconds=2.0, failure_rate=0.0),
]
recommended = recommend_policy_from_measurements(good_measurements)
print(f"Recommended: max_concurrent_requests={recommended.max_concurrent_requests}")
print(f"Reason: {recommended.reason}")

bad_measurements = [
    ConcurrencyMeasurement(concurrency=1, mean_latency_seconds=1.0, p95_latency_seconds=1.2, failure_rate=0.0),
    ConcurrencyMeasurement(concurrency=2, mean_latency_seconds=4.0, p95_latency_seconds=12.0, failure_rate=0.0),
]
recommended_bad = recommend_policy_from_measurements(bad_measurements)
print(f"\nWith a p95 blowup at concurrency=2: recommended stays at {recommended_bad.max_concurrent_requests}")

Default: max_concurrent_requests=1
Reason: default: single unified-memory Mac, not yet measured (see theory doc)

Correctly rejected an unjustified higher concurrency: max_concurrent_requests > 1 requires a specific `reason` citing the measurement that justified it - the default reason is not enough. See recommend_policy_from_measurements() for how to derive one.

--- Deriving a policy from measurements ---
Recommended: max_concurrent_requests=4
Reason: measured: concurrency=4 kept p95 latency at 2.00s (baseline 1.20s at concurrency=1, within 2.0x) with 0% failure rate

With a p95 blowup at concurrency=2: recommended stays at 1


## 5. Real Ollama concurrency measurement (Labs 1-3), if available

Expected to skip on this machine (no local model runtime installed by design).

In [6]:
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_01"))
from ollama_probe import is_ollama_available
import lab_measure_concurrency as concurrency_lab

if is_ollama_available():
    measurements = await concurrency_lab.run_lab("qwen2.5:3b", concurrency_lab.DEFAULT_PROMPT, [1, 2, 4], 8)
    from IPython.display import Markdown, display
    display(Markdown(concurrency_lab.measurements_to_markdown_table(measurements)))
    recommended = recommend_policy_from_measurements(measurements)
    print(f"Recommended: {recommended.max_concurrent_requests} ({recommended.reason})")
else:
    print("SKIPPED: Ollama is not reachable at http://localhost:11434.")
    print("This machine deliberately has no local model runtime installed (see PROGRESS.md).")
    print("On the resourced 32GB Mac: re-run this cell for real 1/2/4-concurrency measurements.")

SKIPPED: Ollama is not reachable at http://localhost:11434.
This machine deliberately has no local model runtime installed (see PROGRESS.md).
On the resourced 32GB Mac: re-run this cell for real 1/2/4-concurrency measurements.


## 6. Take it to the command line

```bash
uv run python scripts/module_06_5/lab_caching_before_after.py     # runs for real, no installs needed
uv run python scripts/module_06_5/lab_measure_concurrency.py --model qwen2.5:3b   # needs Ollama
```

Then fold the output into
[`reports/module_06_5_serving_concurrency_report.md`](../reports/module_06_5_serving_concurrency_report.md).